<a href="https://colab.research.google.com/github/satyam-kr03/GEO/blob/main/GEO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content/drive/My Drive/Seminar/
!ls

/content/drive/My Drive/Seminar
data


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from pathlib import Path

import sys
import cloudpickle
import h5py
from copy import deepcopy
from dataclasses import dataclass
from typing import List
import cvxopt as opt
import numpy as np
from cvxopt import blas, solvers
import time
import logging

from typing import Optional, Tuple, List, Any, Dict
import typing

In [5]:
def save_obj(obj, file_path):
    with open(file_path, "wb") as f:
        r = cloudpickle.dump(obj, f)
    return r


def load_obj(file_path):
    with open(file_path, "rb") as f:
        obj = cloudpickle.load(f)
    return obj

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.mixture import GaussianMixture

class GMMBinary(nn.Module):
    def __init__(self, n_components, input_dim, device=torch.device("cuda" if torch.cuda.is_available() else "cpu")):
        super(GMMBinary, self).__init__()
        self.n_components = n_components
        self.input_dim = input_dim
        self.device_name = device
        self.gmm = GaussianMixture(n_components=n_components, covariance_type='full')

    def fit(self, data):
        # data = data.cpu().numpy()
        self.gmm.fit(data)

    def generate(self, num_samples):
        samples, _ = self.gmm.sample(num_samples)
        samples = torch.tensor(samples).float().to(self.device_name)
        binary_samples = torch.bernoulli(torch.sigmoid(samples))
        return binary_samples.cpu()

    def compute_probability(self, bitstrings_numpy):
        bitstrings = torch.tensor(bitstrings_numpy).float().to(self.device_name)
        log_prob = self.gmm.score_samples(bitstrings.cpu().numpy())
        probabilities = torch.exp(torch.tensor(log_prob)).to(self.device_name)
        return probabilities.cpu()

# Example usage:
# gmm_binary = GMMBinary(n_components=5, input_dim=10)
# gmm_binary.fit(training_data)
# generated_samples = gmm_binary.generate(num_samples=100)
# probabilities = gmm_binary.compute_probability(test_bitstrings)

In [7]:
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import typing
# from typing import List, Tuple

# class Generator(nn.Module):
#     def __init__(self, input_dim, hidden_dim, output_dim):
#         super(Generator, self).__init__()
#         self.model = nn.Sequential(
#             nn.Linear(input_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Linear(hidden_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Linear(hidden_dim, output_dim),
#             nn.Sigmoid()
#         )

#     def forward(self, x):
#         return self.model(x)

# class Discriminator(nn.Module):
#     def __init__(self, input_dim, hidden_dim):
#         super(Discriminator, self).__init__()
#         self.model = nn.Sequential(
#             nn.Linear(input_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Linear(hidden_dim, hidden_dim),
#             nn.ReLU(),
#             nn.Linear(hidden_dim, 1),
#             nn.Sigmoid()
#         )

#     def forward(self, x):
#         return self.model(x)

# class GAN(nn.Module):
#     def __init__(
#         self,
#         input_dim,
#         hidden_dim,
#         noise_dim,
#         device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
#     ):
#         super(GAN, self).__init__()
#         self.generator = Generator(noise_dim, hidden_dim, input_dim)
#         self.discriminator = Discriminator(input_dim, hidden_dim)
#         self.device_name = device
#         self.noise_dim = noise_dim
#         self.to(device)

#     def generate(self, num_samples):
#         noise = torch.randn(num_samples, self.noise_dim).to(self.device_name)
#         samples = self.generator(noise)
#         return samples.cpu()

#     def train_gan(
#         self, data, epochs, batch_size, learning_rate, d_steps=1, g_steps=1
#     ):
#         device = self.device_name
#         self.to(device)
#         optimizer_d = optim.Adam(self.discriminator.parameters(), lr=learning_rate)
#         optimizer_g = optim.Adam(self.generator.parameters(), lr=learning_rate)
#         criterion = nn.BCELoss()

#         for epoch in range(epochs):
#             for i in range(0, len(data), batch_size):
#                 real_data = data[i : i + batch_size].to(device)
#                 batch_size = real_data.size(0)

#                 # Train Discriminator
#                 for _ in range(d_steps):
#                     noise = torch.randn(batch_size, self.noise_dim).to(device)
#                     fake_data = self.generator(noise)

#                     real_labels = torch.ones(batch_size, 1).to(device)
#                     fake_labels = torch.zeros(batch_size, 1).to(device)

#                     optimizer_d.zero_grad()
#                     real_output = self.discriminator(real_data)
#                     fake_output = self.discriminator(fake_data.detach())
#                     loss_d_real = criterion(real_output, real_labels)
#                     loss_d_fake = criterion(fake_output, fake_labels)
#                     loss_d = loss_d_real + loss_d_fake
#                     loss_d.backward()
#                     optimizer_d.step()

#                 # Train Generator
#                 for _ in range(g_steps):
#                     noise = torch.randn(batch_size, self.noise_dim).to(device)
#                     fake_data = self.generator(noise)
#                     fake_labels = torch.ones(batch_size, 1).to(device)

#                     optimizer_g.zero_grad()
#                     fake_output = self.discriminator(fake_data)
#                     loss_g = criterion(fake_output, fake_labels)
#                     loss_g.backward()
#                     optimizer_g.step()

#             print(f"Epoch {epoch+1}, Loss D: {loss_d.item()}, Loss G: {loss_g.item()}")

#     def compute_probability(self, bitstrings_numpy):
#         bitstrings = torch.tensor(bitstrings_numpy).float().to(self.device_name)
#         with torch.no_grad():
#             probabilities = self.discriminator(bitstrings)
#         return probabilities.cpu()

# class ganGenerative(TorchGenerativeModel):
#     def __init__(self, input_dim, hidden_dim, noise_dim, loss_key: str = "loss", lr=0.001) -> None:
#         super().__init__()
#         self._device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#         self._model = GAN(
#             input_dim=input_dim, hidden_dim=hidden_dim, noise_dim=noise_dim, device=self._device
#         )
#         self._input_dim = input_dim
#         self._hidden_dim = hidden_dim
#         self._noise_dim = noise_dim
#         self.optimizer_g = optim.Adam(self._model.generator.parameters(), lr=lr)
#         self.optimizer_d = optim.Adam(self._model.discriminator.parameters(), lr=lr)
#         self.loss_key = loss_key

#     def _generate(
#         self, n_samples: int, random_seed: typing.Optional[int] = None
#     ) -> torch.Tensor:
#         return self._model.generate(n_samples)

#     def _train_on_batch(self, batch: Batch) -> TrainResult:
#         """Train the model on a batch of data.

#         Args:
#             batch (Batch): batch of training data and any additional tensors.

#         Returns:
#             TrainResult: results of training containing model-specific details
#                 Note that TrainResult should be limited to human-in-the-loop analysis
#                 and should not be used in applications
#         """
#         device = self._model.device_name
#         self._model.to(device)
#         data = batch.data.to(device)
#         batch_size = data.size(0)

#         # Train Discriminator
#         noise = torch.randn(batch_size, self._noise_dim).to(device)
#         fake_data = self._model.generator(noise)

#         real_labels = torch.ones(batch_size, 1).to(device)
#         fake_labels = torch.zeros(batch_size, 1).to(device)

#         self.optimizer_d.zero_grad()
#         real_output = self._model.discriminator(data)
#         fake_output = self._model.discriminator(fake_data.detach())
#         loss_d_real = nn.BCELoss()(real_output, real_labels)
#         loss_d_fake = nn.BCELoss()(fake_output, fake_labels)
#         loss_d = loss_d_real + loss_d_fake
#         loss_d.backward()
#         self.optimizer_d.step()

#         # Train Generator
#         noise = torch.randn(batch_size, self._noise_dim).to(device)
#         fake_data = self._model.generator(noise)
#         fake_labels = torch.ones(batch_size, 1).to(device)

#         self.optimizer_g.zero_grad()
#         fake_output = self._model.discriminator(fake_data)
#         loss_g = nn.BCELoss()(fake_output, fake_labels)
#         loss_g.backward()
#         self.optimizer_g.step()

#         return {self.loss_key: loss_g.item()}

#     def compute_probability(self, bitstrings):
#         return self._model.compute_probability(bitstrings_numpy=bitstrings).numpy()

#     @property
#     def sample_size(self) -> Tuple[int, ...]:
#         return self._input_dim

#     @property
#     def _models(self) -> List[nn.Module]:
#         return [self._model]

#     @property
#     def sample_size(self) -> typing.Tuple[int, ...]:
#         return (self._input_dim,)

In [8]:
# for generating random samples with cardinality check (initial dataset)
class RandomCardinalityGenerator:
    def __init__(
        self,
        size,
        cardinality,
        device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
    ):
        self.size = size
        self.cardinality = cardinality
        self.device = device

    def generate(self, n_samples, random_seed=None):
        cardinality = self.cardinality
        # torch.manual_seed(get_random_seed(random_seed))
        if cardinality > self.size:
            raise ValueError(
                "Cardinality should be less than or equal to the bitstring length."
            )

        bitstrings = torch.zeros(
            (n_samples, self.size), dtype=torch.int32, device=self.device
        )

        for i in range(n_samples):
            indices = torch.randperm(self.size, device=self.device)[:cardinality]
            bitstrings[i, indices] = 1

        return torch.tensor(bitstrings.cpu().numpy()).float()

In [9]:
# calculates and returns the cost if called as a function
class PortfolioMulti:
    def __init__(
        self,
        means: np.ndarray,
        covariances: np.ndarray,
        cardinality: int,
        lower_bound: float = 0.0,
        upper_bound: float = 1.0,
        temperature: float = 1.0,
        risk_averse: float = 1.0,
    ):
        assert np.all(covariances == np.transpose(covariances))

        self.means = means
        self.covariances = covariances
        self.lower_bound = lower_bound
        self.upper_bound = upper_bound
        self.market_neutral = False
        self.cardinality = cardinality
        self.weights = np.zeros_like(means)
        self.max_risk = 10.0
        self.temperature = temperature
        self.risk_averse = risk_averse
        self.max_variance = self.max_risk

    def optimal_portfolio(self, returns, covs, risk_averse):
        n = len(returns)
        # Convert to cvxopt matrices
        S = opt.matrix(2 * risk_averse * covs)
        returns = opt.matrix(returns)
        var_max = self.max_risk
        lb = self.lower_bound
        ub = self.upper_bound

        # Create constraints matrices
        # q = opt.matrix(0.0, (n, 1))
        # reversed the factor to include the minus sign in the return term
        q = opt.matrix((risk_averse - 1) * returns)
        I = opt.matrix(np.eye(n))
        G = opt.matrix([-I, I])
        h = opt.matrix([opt.matrix(-lb, (n, 1)), opt.matrix(ub, (n, 1))])
        A = opt.matrix(1.0, (1, n))
        b = opt.matrix([1.0])

        # Calculate efficient frontier weights using quadratic programming
        weights = []
        try:
            sol = solvers.qp(
                S, q, G, h, A, b
            )  # Neither A nor b are provided, as no return level
            # constraint is imposed with risk_averse different from zero
        except:  # noqa
            print("Error solving system of equations.")
            sol = {"status": "unknown"}

        if sol["status"] == "optimal":
            weights.append(sol["x"])
            if np.sum(np.asarray(weights[0]).transpose() < lb) != 0:
                print("weights = ", np.asarray(weights[0]).transpose())
            returnp = [blas.dot(returns, wg) for wg in weights][0]
            varp = [blas.dot(wg, opt.matrix(covs) * wg) for wg in weights][0]
        else:
            weights.append(opt.matrix(var_max, (n, 1)))
            returnp = -var_max
            varp = var_max
        costp = risk_averse * varp - (1 - risk_averse) * returnp

        np.expand_dims([risk_averse, returnp, varp, costp], 0)

        return np.asarray(weights[0]), returnp, varp

    def compute_cost(self, x: np.ndarray) -> float:
        risk_averse = self.risk_averse

        if (
            np.sum(x) == self.cardinality
        ):  # checking if the bitsream meets the cardinality condition
            idxs = list(np.where(x == 1)[0])
            weights, returnp, varp = self.optimal_portfolio(
                self.means[idxs], self.covariances[idxs,][:, idxs], risk_averse,
            )
        else:
            returnp = -self.max_variance
            varp = self.max_variance
            weights = np.zeros((self.cardinality, 1))

        cost = risk_averse * varp - (1 - risk_averse) * returnp

        return {
            "cost": cost,
            "variance": varp,
            "return": returnp,
            "risk_averse": risk_averse,
            "weights": weights,
        }

    def __call__(self, x: np.ndarray) -> float:
        risk_averse = self.risk_averse

        if (
            np.sum(x) == self.cardinality
        ):  # checking if the bitsream meets the cardinality condition
            idxs = list(np.where(x == 1)[0])
            weights, returnp, varp = self.optimal_portfolio(
                self.means[idxs], self.covariances[idxs,][:, idxs], risk_averse,
            )
        else:
            returnp = -self.max_variance
            varp = self.max_variance
            np.zeros((self.cardinality, 1))

        cost = risk_averse * varp - (1 - risk_averse) * returnp

        return cost

    def __str__(self):
        return "PortfolioMulti"

In [10]:
def select_best_init_candidate(geo):
    """replace all self to geo in below code"""
    means = geo.objective.means
    covariances = geo.objective.covariances
    risk_averse = geo.objective.risk_averse
    theta = np.zeros(len(geo.objective.means))
    rouh = np.zeros(len(geo.objective.means))
    for n_qubit in range(len(geo.objective.means)):
        theta[n_qubit] = 1 + (1 - risk_averse) * means[n_qubit]
        rouh[n_qubit] = 1 + (risk_averse * sum(covariances[:][n_qubit])) / len(
            geo.objective.means
        )
    omega = -1 * min(np.min(theta), 0)
    psi = -1 * min(np.min(rouh), 0)
    selected_assets = np.zeros(len(geo.objective.means))
    for n_qubit in range(len(geo.objective.means)):
        selected_assets[n_qubit] = (theta[n_qubit] + omega) / (rouh[n_qubit] + psi)
    sorted_idx = np.argsort(selected_assets)[::-1]
    selected_bitstrings = np.zeros(len(geo.objective.means), dtype=int)
    selected_bitstrings[sorted_idx[0:10]] = 1
    ok = 1
    return selected_bitstrings

In [11]:
import numpy as np

class FrequencySelector:
    def __init__(self, n_selector, top_n):
        self.n_selector = n_selector
        self.top_n = top_n

    def select(self, samples):
        # Flatten the samples to count the frequency of each element
        flattened_samples = np.concatenate(samples)

        # Count the frequency of each unique element
        unique, counts = np.unique(flattened_samples, return_counts=True)
        frequency_dict = dict(zip(unique, counts))

        # Sort the elements by frequency in descending order
        sorted_elements = sorted(frequency_dict.items(), key=lambda item: item[1], reverse=True)

        # Select the top_n most frequent elements
        selected_elements = [element for element, count in sorted_elements[:self.top_n]]

        # Filter the samples to include only those with the selected elements
        selected_samples = [sample for sample in samples if any(elem in sample for elem in selected_elements)]

        return selected_samples

# Example usage:
# frequency_selector = FrequencySelector(n_selector=10, top_n=5)
# selected_samples = frequency_selector.select(samples)

In [12]:
import numpy as np

class GEOStandalone:
    def __init__(
        self,
        objective,
        generator,
        selector,
        fallback_generator,
        observe_given_datapoints,
        constraint_list,
        bitstrings,
        temperature,
        random_seed=None,
    ):
        self.objective = objective
        self.generator = generator
        self.selector = selector
        self.fallback_generator = fallback_generator
        self.observe_given_datapoints = observe_given_datapoints
        self.constraint_list = constraint_list
        self.bitstrings = bitstrings
        self.temperature = temperature
        self.random_seed = random_seed
        if random_seed is not None:
            np.random.seed(random_seed)

    def optimize(self, iterations):
        best_solution = None
        best_score = float('-inf')

        for iteration in range(iterations):
            # Generate new samples
            samples = self.generator.generate(len(self.bitstrings))

            # Apply constraints
            valid_samples = []
            for sample in samples:
                if all(constraint(sample) for constraint in self.constraint_list):
                    valid_samples.append(sample)

            # If no valid samples, use fallback generator
            if not valid_samples:
                valid_samples = self.fallback_generator.generate(len(self.bitstrings))

            # Select the best samples
            selected_samples = self.selector.select(valid_samples)

            # Evaluate the objective function
            for sample in selected_samples:
                score = self.objective(sample)
                if score > best_score:
                    best_score = score
                    best_solution = sample

            # Update bitstrings with selected samples
            self.bitstrings = selected_samples

        return best_solution, best_score

    def _append_new_datapoints(self, new_datapoints, start, end, flags):
        # Assuming flags is a list of booleans indicating whether to append each datapoint
        for i in range(start, end):
            if flags[i]:
                self.bitstrings = np.vstack([self.bitstrings, new_datapoints[i]])



# Example usage:
# geo_standalone = GEOStandalone(
#     objective=objective_function,
#     generator=geo_generator,
#     selector=geo_selector,
#     fallback_generator=RandomCardinalityGenerator(size, cardinality=cardinality),
#     observe_given_datapoints=False,
#     constraint_list=[geo_constraint],
#     bitstrings=geo_init_sample,
#     temperature=temperature,
#     random_seed=random_seed,
# )
# best_solution, best_score = geo_standalone.optimize(iterations=100)

In [13]:
def main_optimization(
    risk_averse=0.0,
    hidden_dim=512,
    index_name="HangSeng",
    n_samples=2000,
    n_epochs=10,
    cardinality=10,
    n_init_samples=2000,
    n_max_optimize_iteration=200,
    n_evaluations=1000,
    n_selector=200,
    random_seed=42,
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu"),
):

    lower_bound: float = 0.01
    upper_bound: float = 1.0
    remove_low_return: bool = False

    data_path = "data/"
    data = h5py.File(data_path + "dataset.h5", "r")
    # Read dataset returns.astype(np.double)
    means = data[index_name]["means"][:].astype(np.double)
    covs = data[index_name]["covs"][:].astype(np.double)
    size = len(means)

    t1 = time.time()
    temperature = np.sqrt(np.mean(covs))
    objective_function = PortfolioMulti(
        means=means,
        covariances=covs,
        cardinality=cardinality,
        lower_bound=lower_bound,
        upper_bound=upper_bound,
        temperature=temperature,
        risk_averse=risk_averse,
    )

    geo_init_sample = RandomCardinalityGenerator(
        size=size, cardinality=cardinality
    ).generate(n_init_samples, random_seed=random_seed)
    geo_constraint = None  # CardinalityConstraint(cardinality)
    gmm_binary = GMMBinary(n_components=5, input_dim=10)
    initial_data = np.random.rand(n_samples, 10)
    gmm_binary.fit(initial_data)
    geo_generator = gmm_binary
    # geo_selector = ProbabilitySelector(n_selector)
    geo_selector = FrequencySelector(n_selector, n_selector)

    solver = GEOStandalone(
        objective=objective_function,
        generator=geo_generator,
        selector=geo_selector,
        fallback_generator=RandomCardinalityGenerator(size, cardinality=cardinality),
        observe_given_datapoints=False,
        constraint_list=[geo_constraint],
        bitstrings=geo_init_sample,
        temperature=temperature,
        # random_seed=random_seed,
    )

    best_sample = select_best_init_candidate(solver)
    solver._append_new_datapoints(
        np.array([best_sample], dtype=np.float32), 0, 1, [False]
    )

    solution = solver.optimize(
        iterations=n_max_optimize_iteration,
    )
    t2 = time.time()

    data = {
        "solution": solution,
        # "solver.idata": solver.idata,
        "solver.idata": solver.datapoints,
        "nade": geo_generator,
        "solver": solver,
        "time": t2 - t1,
    }
    return data

In [14]:
if __name__ == "__main__":
    start = 0
    end = 50
    n_segmentation = 10
    index_name = "HangSeng"
    hidden_dim = 512
    n_selector = 200
    ex_no = 8
    results = []
    i = start
    n_epochs = 30
    n_samples = 50000
    n_init_samples = 1500
    n_max_optimize_iteration = 200
    random_seed = 1

    indexs = {
        "HangSeng": 31,
        "FTSE100": 89,
        "SP100": 98,
        "XU100": 99,
        "DAX100": 85,
        "Nikkei225": 225,
        "XU030": 30,
    }
    # print(f"start {start}, end {end},index name {index_name}")
    # time.sleep(10)

    Path(f"data/{index_name}").mkdir(parents=True, exist_ok=True)
    for risk_averse in np.linspace(0.0, 1.0, n_segmentation).tolist()[start:end]:
        print(f"risk_averse_step:{i}, risk_averse:{risk_averse}")
        result = main_optimization(
            index_name=index_name,
            hidden_dim=hidden_dim,
            n_epochs=n_epochs,
            n_samples=n_samples,
            n_init_samples=n_init_samples,
            risk_averse=risk_averse,
            cardinality=10,
            n_evaluations=indexs[index_name] * 1000,
            n_selector=n_selector,
            n_max_optimize_iteration=n_max_optimize_iteration,
            random_seed=random_seed,
        )
        result_in_list = [risk_averse, result]
        save_obj(
            result_in_list,
            f"data/{index_name}/{i}_{risk_averse}_{index_name}_ex_no_{ex_no}.dill",
        )
        results.append(result_in_list)
        i += 1

    # save_obj(results, f"data/{index_name}/full_results_{index_name}_ex_no_{ex_no}.dill")


risk_averse_step:0, risk_averse:0.0


TypeError: 'NoneType' object is not callable

In [ ]:
!ls

In [ ]:
# import numpy as np
# from sklearn.mixture import GaussianMixture
# from scipy.optimize import minimize

# class GEO:
#     def __init__(self, n_assets, cardinality, n_iterations=100, n_samples=50):
#         self.n_assets = n_assets
#         self.cardinality = cardinality
#         self.n_iterations = n_iterations
#         self.n_samples = n_samples
#         self.gmm = GaussianMixture(n_components=2, covariance_type='full')
#         self.best_solution = None
#         self.best_fitness = float('inf')

#         # Initialize the GMM with random data
#         initial_data = np.random.rand(n_samples, n_assets)
#         self.gmm.fit(initial_data)

#     def objective_function(self, x):
#         # Placeholder for the portfolio optimization objective function
#         # This should be replaced with the actual objective function
#         return np.sum((x - 0.5)**2)

#     def generate_samples(self):
#         samples = self.gmm.sample(self.n_samples)[0]
#         samples = (samples > 0.5).astype(int)
#         samples = np.array([self._adjust_cardinality(s) for s in samples])
#         return samples

#     def _adjust_cardinality(self, sample):
#         current_cardinality = np.sum(sample)
#         if current_cardinality > self.cardinality:
#             indices = np.where(sample == 1)[0]
#             to_remove = np.random.choice(indices, current_cardinality - self.cardinality, replace=False)
#             sample[to_remove] = 0
#         elif current_cardinality < self.cardinality:
#             indices = np.where(sample == 0)[0]
#             to_add = np.random.choice(indices, self.cardinality - current_cardinality, replace=False)
#             sample[to_add] = 1
#         return sample

#     def update_model(self, samples, fitnesses):
#         best_samples = samples[np.argsort(fitnesses)[:self.n_samples//2]]
#         self.gmm.fit(best_samples)

#     def optimize(self):
#         for _ in range(self.n_iterations):
#             samples = self.generate_samples()
#             fitnesses = np.array([self.objective_function(s) for s in samples])

#             best_idx = np.argmin(fitnesses)
#             if fitnesses[best_idx] < self.best_fitness:
#                 self.best_solution = samples[best_idx]
#                 self.best_fitness = fitnesses[best_idx]

#             self.update_model(samples, fitnesses)

#         return self.best_solution, self.best_fitness

# # Usage example
# n_assets = 100
# cardinality = 50
# geo_solver = GEO(n_assets, cardinality)
# best_solution, best_fitness = geo_solver.optimize()
# print(f"Best solution: {best_solution}")
# print(f"Best fitness: {best_fitness}")
